# Mini-LLM para a HP 50g (Edição Hiper-Focada)

Treino de um modelo de linguagem minúsculo focado na memorização de fatos exatos
em português, para rodar em C (HPGCC) na HP 50g.


In [ ]:
# 1) PREPARAR: GPU, Google Drive, código e atalhos
RAMO = "main"  #@param {type:"string"}
import os, subprocess, codecs, json, glob
def rodar(comando):
    print("$", comando, flush=True)
    p = subprocess.Popen(comando, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env=dict(os.environ, PYTHONUNBUFFERED="1"))
    t = codecs.getincrementaldecoder("utf-8")("replace")
    while pedaco := p.stdout.read1(65536):
        print(t.decode(pedaco), end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"o comando falhou (código {p.returncode}): {comando}")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (só processador)")
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/LLM_Laika"
HP = f"{DRIVE}/hp50g"
os.makedirs(HP, exist_ok=True)
if not os.path.exists("/content/LLM_Laika"):
    rodar("git clone -q https://github.com/BrunoDevPro/LLM_Laika.git /content/LLM_Laika")
os.chdir("/content/LLM_Laika")
rodar(f"git fetch -q origin {RAMO} && git checkout -q -f -B {RAMO} FETCH_HEAD")
rodar("pip install -q tokenizers")
LOCAL = "/content/hp50g_dados"
def dados_locais():
    os.makedirs(LOCAL, exist_ok=True)
    return LOCAL


In [ ]:
# 2) DADOS E TOKENIZADOR (Integrado na Célula 6)
print("A preparação de dados agora roda direto na Célula 6.")


In [ ]:
# 3) VARREDURA CURTA
print("Pulado.")


In [ ]:
# 4) VARREDURA DE DADOS
print("Pulado.")


In [ ]:
# 5) PROFESSOR
print("Pulado.")


In [ ]:
# 6) PREPARAÇÃO DOS DADOS E TREINO FINAL DO ZERO
ARQUIVO_FATOS = "/content/drive/MyDrive/LLM_Laika/hp50g/meus_fatos.txt" #@param {type:"string"}
VARIANTE = "A"  #@param {type:"string"}
TAXA = 3e-3  #@param {type:"number"}
TOKENS = 5e7  #@param {type:"number"}
DO_ZERO = True #@param {type:"boolean"}
NOME = "A_fatos"  #@param {type:"string"}

import json, random, struct, re, os, shutil
import numpy as np
from pathlib import Path
from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers

print("--- INICIANDO PREPARAÇÃO DOS DADOS ---")
SAIDA = Path("/content/hp50g_dados")
SAIDA.mkdir(parents=True, exist_ok=True)
SAIDA_DRIVE = Path(f"{HP}/dados")
SAIDA_DRIVE.mkdir(parents=True, exist_ok=True)

ESPECIAIS = ["<|endoftext|>", "<|user|>", "<|assistant|>", "<|end|>"]
EOT, USER, ASSISTANT, END = ESPECIAIS

fatos = []
if os.path.exists(ARQUIVO_FATOS):
    with open(ARQUIVO_FATOS, "r", encoding="utf-8") as f:
        for linha in f:
            if "|" in linha:
                p, r = linha.split("|", 1)
                fatos.append((p.strip(), r.strip()))
    print(f"Carregados {len(fatos)} fatos únicos.")
else:
    raise FileNotFoundError(f"ERRO: O arquivo '{ARQUIVO_FATOS}' não foi encontrado.")

texto_puro = " ".join([f"{p} {r}" for p, r in fatos])
amostra = SAIDA / "_amostra_tok.txt"
amostra.write_text(texto_puro, encoding="utf-8")

tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Split(Regex(r" ?[^\s]+|\s+"), behavior="isolated"),
    pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
])
tok.decoder = decoders.ByteLevel()
treinador = trainers.BpeTrainer(vocab_size=1024, special_tokens=ESPECIAIS, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
tok.train([str(amostra)], treinador)
tok.save(str(SAIDA / "tokenizador.json"))
amostra.unlink()

dataset = fatos * 2000
random.shuffle(dataset)
corte = int(len(dataset) * 0.95)
treino, val = dataset[:corte], dataset[corte:]

# Salva dialogos_val.txt (exigido pelo simulador.py na Célula 7)
with open(SAIDA / "dialogos_val.txt", "w", encoding="utf-8") as f_val:
    for p, r in val[:200]:
        f_val.write(f"{USER} {p} {ASSISTANT} {r} {END}\n{EOT}\n")

meta = {"vocab": tok.get_vocab_size(), "especiais": {e: tok.token_to_id(e) for e in ESPECIAIS}, "fontes": {}}
eot_id = tok.token_to_id(EOT)

for fonte_nome in ["dialogos", "texto"]:
    meta["fontes"][fonte_nome] = {}
    for split_nome, split_dados in [("treino", treino), ("val", val)]:
        n_tok = 0
        with open(SAIDA / f"{fonte_nome}_{split_nome}.bin", "wb") as f_out:
            if fonte_nome == "dialogos":
                lote = [f"{USER} {p} {ASSISTANT} {r} {END}" for p, r in split_dados]
            else:
                lote = [f"{r}" for _, r in split_dados]
            
            for enc in tok.encode_batch(lote):
                ids = np.asarray(enc.ids + [eot_id], dtype=np.uint16)
                f_out.write(ids.tobytes())
                n_tok += len(ids)
        meta["fontes"][fonte_nome][split_nome] = {"tokens": n_tok, "docs": len(split_dados)}
    meta["fontes"][fonte_nome]["caracteres_por_token"] = 3.5

(SAIDA / "meta.json").write_text(json.dumps(meta, indent=1, ensure_ascii=False), encoding="utf-8")

j = json.loads((SAIDA / "tokenizador.json").read_text(encoding="utf-8"))
vocab = j["model"]["vocab"]
especiais_dict = {t["id"]: t["content"] for t in j["added_tokens"]}

def bytes_do_token(t):
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    tabela = dict(zip(bs, map(chr, cs)))
    inv = {v: k for k, v in tabela.items()}
    return bytes(inv[ch] for ch in t)

n_tokens = max(vocab.values()) + 1
for t in j["added_tokens"]: n_tokens = max(n_tokens, t["id"] + 1)
tokens_bytes = [b""] * n_tokens
for texto, i in vocab.items():
    tokens_bytes[i] = especiais_dict[i].encode() if i in especiais_dict else bytes_do_token(texto)
for i, t in especiais_dict.items(): tokens_bytes[i] = t.encode()

fusoes = []
for f in j["model"]["merges"]:
    a, b = f.split(" ") if isinstance(f, str) else f
    fusoes.append((vocab[a], vocab[b], vocab[a + b]))

with open(SAIDA / "vocab.bin", "wb") as f_out:
    f_out.write(b"HPVB" + struct.pack("<HHH", len(tokens_bytes), len(fusoes), len(especiais_dict)))
    for t in tokens_bytes: f_out.write(struct.pack("<B", len(t)) + t)
    for a_, b_, r in fusoes: f_out.write(struct.pack("<HHH", a_, b_, r))
    for i in sorted(especiais_dict): f_out.write(struct.pack("<H", i))

# Copia os arquivos gerados também para o Drive por segurança
for arq in SAIDA.glob("*"):
    shutil.copy(arq, SAIDA_DRIVE / arq.name)

print("--- DADOS PREPARADOS. INICIANDO TREINO ---")

dl = str(SAIDA)
nome = NOME.strip() or f"{VARIANTE}_fatos"
flag_do_zero = " --do-zero" if DO_ZERO else ""

comando = (f"python hp50g/treino.py --variante {VARIANTE} --lr {TAXA} "
           f"--tokens {TOKENS:.0f} --dados {dl} "
           f"--saida {HP}/final/{nome} --compile{flag_do_zero}")

rodar(comando)


In [ ]:
# 7) EXPORTAR E RELATÓRIO
EXECUCOES = "A_fatos"  #@param {type:"string"}
JANELAS = 200  #@param {type:"integer"}
nomes = [n.strip() for n in EXECUCOES.split(",") if n.strip()] or sorted(os.listdir(f"{HP}/final"))
pacotes = []
for n in nomes:
    pasta = next(p for p in (f"{HP}/final/{n}", f"{HP}/varredura/{n}", f"{HP}/dados_x/{n}") if os.path.isdir(p))
    rodar(f"python hp50g/simulador.py exportar --ckpt {pasta}/melhor.pt --dados /content/hp50g_dados --saida {HP}/export/{n} "
          f"--nome {n} --janelas {JANELAS}")
    pacotes.append(f"{HP}/export/{n}")
rodar(f"python hp50g/simulador.py relatorio --pacotes {' '.join(pacotes)} --saida {HP}/export/relatorio.md")
